# 🚩 Sprint 1 · Checkpoint 04 — Newton & Heavy Artillery

**Companion ao Boyd (Cap 9 — Newton's Method) + Weinberger (Lec 6 Proof, Lec 12 Newton).**

Até aqui só usaste a **primeira derivada** (gradiente). Newton dá um salto de qualidade: usa também a **segunda derivada** (curvatura, Hessiana) para escolher o passo. Em problemas quadráticos converge em **1 passo**.

A Hessiana é cara em alta dimensão ($O(d^2)$ memória, $O(d^3)$ inversão). Daí os otimizadores adaptativos modernos — **RMSProp**, **Adam**, **AdamW** — que aproximam o efeito de Newton sem nunca formar a Hessiana explicitamente.

**Como trabalhar:**
- Cada `# TODO` é uma micro-task. Substitui `...` ou completa o esqueleto.
- A célula corre uma `assert` — se falhar, lê a mensagem e corrige.
- Trabalha em `solutions/local.ipynb`.

## ⚙️ Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)

def check(cond, msg='falhou'):
    assert cond, f'❌ {msg}'
    print('✅', msg)

## §1 — Gradiente numérico (central difference)

Antes de codares Newton precisas de uma maneira de **estimar gradientes sem fazer cálculos analíticos**. Em ML real, vais ter funções complicadas onde derivar à mão é impossível ou propenso a erros. A diferença finita central:

$$ \frac{\partial f}{\partial x_i} \approx \frac{f(x + h\, e_i) - f(x - h\, e_i)}{2 h} $$

Onde $e_i$ é o vetor da base canónica (zeros com 1 na posição $i$).

**Erro de truncatura:** $O(h^2)$ — daí "central" ser melhor que "forward" ($O(h)$).
**Erro numérico:** cresce quando $h$ é demasiado pequeno (cancelamento). Sweet spot: $h \sim 10^{-5}$ para `float64`.

### Task 1.1 — Implementa `numerical_grad`

In [ ]:
def numerical_grad(f, x, h=1e-5):
    """Gradiente de f num ponto x, via central difference."""
    x = np.asarray(x, dtype=float)
    g = np.zeros_like(x)
    for i in range(len(x)):
        x_plus = x.copy(); x_plus[i] += h
        x_minus = x.copy(); x_minus[i] -= h
        # TODO: g[i] = (f(x_plus) - f(x_minus)) / (2 * h)
        ...
    return g

# Função teste 1: f(x, y) = x² + 2y². Gradiente analítico = (2x, 4y)
def f1(x):
    return x[0]**2 + 2*x[1]**2

g_num = numerical_grad(f1, [3.0, 1.5])
g_analy = np.array([6.0, 6.0])  # (2*3, 4*1.5)
check(np.allclose(g_num, g_analy, atol=1e-6),
      f'gradiente numérico ≈ analítico? num={g_num}, analy={g_analy}')

# Função teste 2: Rosenbrock 2D. f(x,y) = (1-x)² + 100(y-x²)²
# Gradiente: (-2(1-x) - 400 x (y - x²), 200(y - x²))
def rosenbrock(x):
    return (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2

x_test = np.array([0.5, 0.3])
g_num_ros = numerical_grad(rosenbrock, x_test)
# Analítico:
#  ∂/∂x = -2(1-x) - 400 x (y - x²) = -2(1-0.5) - 400·0.5·(0.3 - 0.25) = -1 - 10 = -11
#  ∂/∂y = 200(y - x²) = 200·(0.3 - 0.25) = 10
g_ros_analy = np.array([-11.0, 10.0])
check(np.allclose(g_num_ros, g_ros_analy, atol=1e-3),
      f'gradiente Rosenbrock: num={g_num_ros}, analy={g_ros_analy}')

## §2 — Hessiana numérica

A Hessiana é a matriz de **segundas derivadas**:

$$ H_{ij} = \frac{\partial^2 f}{\partial x_i \partial x_j} $$

Para $f: \mathbb{R}^d \to \mathbb{R}$, a Hessiana é $d \times d$ e simétrica (Schwarz). Em problemas convexos é **semi-definida positiva** (todos os valores próprios $\ge 0$).

**Estimação numérica via diferenças finitas:** aplica `numerical_grad` a cada componente do gradiente. Custa $O(d^2)$ avaliações de $f$.

$$ H_{ij} \approx \frac{\partial g_i}{\partial x_j} \approx \frac{g_i(x + h e_j) - g_i(x - h e_j)}{2 h} $$

### Task 2.1 — Implementa `numerical_hessian`

In [ ]:
def numerical_hessian(f, x, h=1e-4):
    """Hessiana de f num ponto x, via diferenças finitas em cima do gradiente."""
    x = np.asarray(x, dtype=float)
    n = len(x)
    H = np.zeros((n, n), dtype=float)
    for j in range(n):
        x_plus = x.copy(); x_plus[j] += h
        x_minus = x.copy(); x_minus[j] -= h
        # TODO: g_plus = numerical_grad(f, x_plus, h=h)
        # TODO: g_minus = numerical_grad(f, x_minus, h=h)
        # TODO: H[:, j] = (g_plus - g_minus) / (2 * h)
        ...
    # Simetriza para reduzir erro numérico
    return 0.5 * (H + H.T)

# Hessiana de f1(x,y) = x² + 2y² é diag(2, 4) (constante!)
H1 = numerical_hessian(f1, [3.0, 1.5])
H1_analy = np.diag([2.0, 4.0])
check(np.allclose(H1, H1_analy, atol=1e-3),
      f'Hessiana de x²+2y² deve ser diag(2,4), obteve\n{H1}')

# Simetria
check(np.allclose(H1, H1.T, atol=1e-9), 'Hessiana deve ser simétrica')

# Definida positiva: todos os valores próprios > 0
eigvals = np.linalg.eigvalsh(H1)
check(np.all(eigvals > 0), f'eigenvalues devem ser > 0 (DP), obteve {eigvals}')

## §3 — Método de Newton

A receita: aproxima $f$ localmente por uma quadrática (expansão de Taylor de 2ª ordem):

$$ f(x + p) \approx f(x) + g^T p + \tfrac{1}{2} p^T H p $$

Minimizar essa aproximação em $p$ dá o **passo de Newton**:

$$ p = -H^{-1} g $$

Update: $x_{t+1} = x_t + p = x_t - H^{-1} g$.

**O milagre quadrático:** se $f$ é exatamente quadrática (Hessiana constante), Newton converge em **1 passo** — ainda que estejas a $10^{10}$ de distância. GD precisaria de muitos passos.

**Custo:** inverter $H$ é $O(d^3)$. Para $d = 10^6$ (deep nets), inviável. Daí Adam et al.

### Task 3.1 — Newton em 1 passo numa quadrática

In [ ]:
def newton_step(grad_fn, hess_fn, x):
    """Calcula x_new = x - H⁻¹ g num ponto."""
    g = grad_fn(x)
    H = hess_fn(x)
    # TODO: p = -np.linalg.solve(H, g)  (mais estável que np.linalg.inv(H) @ g)
    # TODO: devolve x + p
    return ...

# Em f1 = x² + 2y², a partir de qualquer x, Newton chega ao mínimo (0,0) em 1 passo.
def grad_f1(x):
    return numerical_grad(f1, x)

def hess_f1(x):
    return numerical_hessian(f1, x)

x_start = np.array([10.0, -7.0])
x_after = newton_step(grad_f1, hess_f1, x_start)
print(f'após 1 passo de Newton: {x_after}')
check(np.linalg.norm(x_after) < 1e-3,
      f'em quadrática, Newton chega ao mínimo em 1 passo, obteve {x_after}')

### Task 3.2 — Newton com line search em Rosenbrock

Rosenbrock NÃO é quadrática (é não-convexa local, mas tem mínimo único em $(1, 1)$). Newton puro pode falhar — o passo $p = -H^{-1}g$ pode "saltar de mais" ou apontar para uma direção que aumenta $f$ se a Hessiana não for definida positiva localmente.

A solução clássica é **damped Newton** com **backtracking line search** (Armijo simplificado):

1. Calcula a direção de Newton $p = -H^{-1} g$.
2. Se $p$ não aponta para descida (ou seja, $p^T g \ge 0$), usa $p = -g$ em vez disso.
3. Procura $\alpha \in (0, 1]$ tal que $f(x + \alpha p) < f(x)$, dividindo $\alpha$ por 2 até funcionar.
4. Update: $x \gets x + \alpha p$.

Isto é exatamente o que `scipy.optimize.minimize(method='Newton-CG')` faz por dentro.

In [ ]:
def newton_iter(f, grad_fn, hess_fn, x0, n_iter):
    """Damped Newton com backtracking line search."""
    x = np.asarray(x0, dtype=float).copy()
    history = [x.copy()]
    for _ in range(n_iter):
        g = grad_fn(x)
        H = hess_fn(x)
        try:
            p = -np.linalg.solve(H, g)
        except np.linalg.LinAlgError:
            p = -g  # singular → desce no gradiente
        # Se a direção de Newton não desce, usa -g (steepest descent local)
        if np.dot(p, g) >= 0:
            p = -g
        # TODO: line search por halving — começa com alpha=1
        # TODO: enquanto f(x + alpha*p) >= f(x) e alpha > 1e-10, faz alpha /= 2
        alpha = 1.0
        f_curr = f(x)
        while alpha > 1e-10 and f(x + alpha * p) >= f_curr:
            alpha *= 0.5
        # TODO: x = x + alpha * p
        ...
        history.append(x.copy())
    return np.array(history)

# Compara em Rosenbrock a partir de (-1.2, 1) — o ponto inicial canónico
def grad_ros(x):
    return numerical_grad(rosenbrock, x)

def hess_ros(x):
    return numerical_hessian(rosenbrock, x)

x0 = np.array([-1.2, 1.0])

# Damped Newton — convergência típica em <100 iter para Rosenbrock
hist_newton = newton_iter(rosenbrock, grad_ros, hess_ros, x0, n_iter=100)
final_newton = hist_newton[-1]
print(f'Newton final: {final_newton}, f={rosenbrock(final_newton):.6f}')

# GD vanilla com lr seguro — esperado: muito lento em Rosenbrock
def gd(x0, grad_fn, lr, n_iter):
    x = np.asarray(x0, dtype=float).copy()
    history = [x.copy()]
    for _ in range(n_iter):
        x = x - lr * grad_fn(x)
        history.append(x.copy())
    return np.array(history)

hist_gd = gd(x0, grad_ros, lr=0.001, n_iter=100)
final_gd = hist_gd[-1]
print(f'GD final (100 iter): {final_gd}, f={rosenbrock(final_gd):.6f}')

check(np.linalg.norm(final_newton - np.array([1.0, 1.0])) < 1e-3,
      f'Newton com line search deve atingir (1,1) a <0.001, obteve {final_newton}')
check(rosenbrock(final_newton) < rosenbrock(final_gd),
      f'Newton com line search deve bater GD: f_N={rosenbrock(final_newton):.6f}, f_GD={rosenbrock(final_gd):.6f}')

## §4 — RMSProp: fix ao AdaGrad

AdaGrad (CP03) acumula gradientes ao quadrado **sem esquecer** — eventualmente os passos morrem. RMSProp (Hinton, lecture notes 2012) substitui a soma por uma **média móvel exponencial**:

$$ s_{t+1} = \rho\, s_t + (1-\rho)\, g_t \odot g_t \\ x_{t+1} = x_t - \frac{\eta}{\sqrt{s_{t+1}} + \epsilon} \odot g_t $$

`ρ ∈ [0.9, 0.999]`. Os gradientes antigos decaem exponencialmente — o "memória" tem escala $1/(1-\rho)$ passos.

### Task 4.1 — Implementa RMSProp

In [ ]:
def rmsprop(x0, grad_fn, lr, rho=0.9, n_iter=200, eps=1e-8):
    x = np.asarray(x0, dtype=float).copy()
    s = np.zeros_like(x)
    history = [x.copy()]
    for _ in range(n_iter):
        g = grad_fn(x)
        # TODO: s = rho * s + (1 - rho) * g * g
        # TODO: x = x - lr * g / (np.sqrt(s) + eps)
        ...
        history.append(x.copy())
    return np.array(history)

# Testa em Rosenbrock
hist_rms = rmsprop(x0, grad_ros, lr=0.05, rho=0.9, n_iter=500)
final_rms = hist_rms[-1]
print(f'RMSProp final: {final_rms}, f={rosenbrock(final_rms):.6f}')
check(rosenbrock(final_rms) < 0.5,
      f'RMSProp deve atingir f<0.5 em 500 iter, obteve {rosenbrock(final_rms):.4f}')

## §5 — Adam: o "padrão da indústria"

Adam (Kingma & Ba, 2014) combina momentum e RMSProp. Mantém duas médias móveis: o **primeiro momento** (média) e o **segundo momento** (variância).

$$ m_{t+1} = \beta_1\, m_t + (1 - \beta_1)\, g_t \\ v_{t+1} = \beta_2\, v_t + (1 - \beta_2)\, g_t \odot g_t $$

**Bias correction** (importante nos primeiros passos quando $m, v$ ainda começam em zero):

$$ \hat{m} = \frac{m_{t+1}}{1 - \beta_1^{t+1}},\quad \hat{v} = \frac{v_{t+1}}{1 - \beta_2^{t+1}} $$

Update:

$$ x_{t+1} = x_t - \eta\, \frac{\hat{m}}{\sqrt{\hat{v}} + \epsilon} $$

Defaults: `β1=0.9, β2=0.999, lr=0.001, eps=1e-8`. Estes números aparecem em ~todos os papers de deep learning.

### Task 5.1 — Implementa Adam

In [ ]:
def adam(x0, grad_fn, lr=0.001, beta1=0.9, beta2=0.999, n_iter=200, eps=1e-8):
    x = np.asarray(x0, dtype=float).copy()
    m = np.zeros_like(x)
    v = np.zeros_like(x)
    history = [x.copy()]
    for t in range(n_iter):
        g = grad_fn(x)
        # TODO: m = beta1 * m + (1 - beta1) * g
        # TODO: v = beta2 * v + (1 - beta2) * g * g
        # Bias correction (t começa em 0, então usamos t+1)
        # TODO: m_hat = m / (1 - beta1 ** (t + 1))
        # TODO: v_hat = v / (1 - beta2 ** (t + 1))
        # TODO: x = x - lr * m_hat / (np.sqrt(v_hat) + eps)
        ...
        history.append(x.copy())
    return np.array(history)

# Adam em Rosenbrock — defaults bem afinados
hist_adam = adam(x0, grad_ros, lr=0.01, n_iter=2000)
final_adam = hist_adam[-1]
print(f'Adam final: {final_adam}, f={rosenbrock(final_adam):.6f}')
check(rosenbrock(final_adam) < 0.05,
      f'Adam deve aproximar (1,1) em 2000 iter, obteve f={rosenbrock(final_adam):.4f}')

# Sanity: bias correction faz diferença nos primeiros 50 passos.
# Sem bias correction, m_hat ≈ (1-β1)*g = 0.1*g, demasiado pequeno.
# Compara o primeiro passo com e sem bias correction.
m1 = (1 - 0.9) * grad_ros(x0)  # primeiro passo de m
m1_corrected = m1 / (1 - 0.9 ** 1)  # = m1 / 0.1 = 10 * m1
check(np.linalg.norm(m1_corrected) > np.linalg.norm(m1) * 5,
      'bias correction amplia o passo inicial em ~10× (essencial nas primeiras iter)')

## §6 — AdamW: weight decay decoupled

Adam clássico mistura **regularização L2** (penalty $\lambda \|x\|^2$ adicionada à perda) com weight decay (subtrair $\lambda x$ ao update). Em Adam, esta confusão **rouba a calibração** dos passos adaptativos: o decay efetivo escala com $\sqrt{v}$.

**AdamW** (Loshchilov & Hutter, 2019) separa: o weight decay é aplicado *diretamente* ao parâmetro, fora do passo adaptativo.

$$ x_{t+1} = x_t - \eta\, \frac{\hat{m}}{\sqrt{\hat{v}} + \epsilon} - \eta\, \lambda\, x_t $$

Resultado: regularização funciona como esperado e a maioria dos papers de transformers (BERT, ViT, etc.) usa AdamW por defeito.

### Task 6.1 — Implementa AdamW e mostra o efeito do `weight_decay`

In [ ]:
def adamw(x0, grad_fn, lr=0.001, beta1=0.9, beta2=0.999, weight_decay=0.0,
          n_iter=200, eps=1e-8):
    x = np.asarray(x0, dtype=float).copy()
    m = np.zeros_like(x)
    v = np.zeros_like(x)
    history = [x.copy()]
    for t in range(n_iter):
        g = grad_fn(x)
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g * g
        m_hat = m / (1 - beta1 ** (t + 1))
        v_hat = v / (1 - beta2 ** (t + 1))
        # TODO: passo adaptativo + weight decay independente
        # TODO: x = x - lr * m_hat / (np.sqrt(v_hat) + eps) - lr * weight_decay * x
        ...
        history.append(x.copy())
    return np.array(history)

# Função teste: f(x,y) = (x-3)² + (y-4)² (mínimo em (3, 4), distante da origem)
def f_offset(x):
    return (x[0] - 3)**2 + (x[1] - 4)**2

def grad_offset(x):
    return numerical_grad(f_offset, x)

# Sem weight decay: converge para (3, 4)
hist_no_wd = adamw([0.0, 0.0], grad_offset, lr=0.1, weight_decay=0.0, n_iter=200)
print(f'AdamW wd=0:    final = {hist_no_wd[-1]}')
check(np.linalg.norm(hist_no_wd[-1] - np.array([3.0, 4.0])) < 0.1,
      f'sem decay, deve convergir para (3,4), obteve {hist_no_wd[-1]}')

# Com weight decay alto: puxa a solução de volta para a origem
# Mínimo deslocado: minimizar (x-3)² + (y-4)² + λ‖x‖² → x* ≈ (3λ⁻¹/(1+λ⁻¹), 4λ⁻¹/(1+λ⁻¹)) na escala lr
# Direção: cada componente final fica menor que sem decay
hist_wd = adamw([0.0, 0.0], grad_offset, lr=0.1, weight_decay=0.5, n_iter=200)
print(f'AdamW wd=0.5:  final = {hist_wd[-1]}')
check(np.linalg.norm(hist_wd[-1]) < np.linalg.norm(hist_no_wd[-1]),
      f'com weight decay, ||x_final|| deve ser menor (puxa para 0)')

## 🏁 Newton — fechado

Se chegaste aqui sem `AssertionError`, dominaste:

1. **Gradiente numérico** (central difference) — base para verificar gradientes analíticos em qualquer projeto.
2. **Hessiana numérica** — matriz simétrica, definida positiva em pontos de mínimo convexo.
3. **Método de Newton** — minimiza quadráticas em 1 passo, com fallback para GD em zonas de Hessiana não-DP.
4. **RMSProp** — média móvel de $g^2$, fix ao AdaGrad.
5. **Adam** — momentum + RMSProp + bias correction; o default de muitos papers.
6. **AdamW** — weight decay desacoplado, padrão moderno em transformers.

**Perguntas de fecho:**
- Por que é que Newton converge em 1 passo numa função quadrática? (Pista: a expansão de Taylor é exata.)
- O custo computacional do passo de Newton em $\mathbb{R}^d$ é $O(d^3)$. Em ML, $d$ pode ser $10^6$+. Como é que Adam/RMSProp evitam este custo e qual é a aproximação que estão a fazer?
- O que muda exatamente entre Adam e AdamW? Quando importa?

**Próximo checkpoint:** [05 — Showdown](../05_Checkpoint_Showdown/) — capstone. Pões os 7 otimizadores que codaste a competir na função de Rosenbrock e produzes um relatório.